In [ ]:
!pip install catboost

In [1]:
import numpy as np                        # for numerical operations and random data
import pandas as pd                       # for creating and managing DataFrame
from catboost import CatBoostClassifier   # main CatBoost model
from sklearn.model_selection import train_test_split   # to split data into train/test
from sklearn.metrics import (
    accuracy_score,           # overall correct predictions %
    roc_auc_score,            # area under ROC curve — main metric
    classification_report,    # precision, recall, f1 for each class
    confusion_matrix          # shows TP, TN, FP, FN breakdown
)
import warnings                           # to suppress unnecessary warnings
warnings.filterwarnings('ignore')         # hide all warnings from output

In [2]:
# ─────────────────────────────────────────
# Step 1: Create Dataset Manually
# ─────────────────────────────────────────

np.random.seed(42)       # fix random seed so same data generates every run
n = 1000                 # total number of employee records to generate

# ── 5 Numerical Features ──────────────────

age = np.random.randint(22, 60, n)
# employee age between 22 and 60 years

monthly_salary = np.random.randint(20000, 120000, n)
# monthly salary in rupees between 20k and 1.2 lakh

years_at_company = np.random.randint(0, 20, n)
# how many years employee has been with the company

work_hours_per_week = np.random.randint(35, 70, n)
# weekly working hours — high = more stress = more attrition risk

performance_score = np.random.uniform(1.0, 5.0, n).round(1)
# performance rating given by manager — scale 1.0 to 5.0


# ── 3 Categorical Features ────────────────

department = np.random.choice(
    ['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'],
    size=n
)
# which department the employee belongs to

job_level = np.random.choice(
    ['Junior', 'Mid', 'Senior', 'Lead', 'Manager'],
    size=n
)
# seniority level of the employee

work_mode = np.random.choice(
    ['Remote', 'Hybrid', 'On-site'],
    size=n,
    p=[0.3, 0.4, 0.3]    # 30% remote, 40% hybrid, 30% on-site (realistic distribution)
)
# where the employee works from


# ── Target Variable (Attrition) ───────────

# Create realistic attrition logic — not fully random
# Higher attrition chances when: low salary, high hours, low performance
attrition_prob = (
    0.1                                              # base probability 10%
    + (work_hours_per_week > 55) * 0.15             # +15% if overworked
    + (monthly_salary < 40000) * 0.20               # +20% if underpaid
    + (performance_score < 2.5) * 0.10              # +10% if low performer
    + (years_at_company < 2) * 0.10                 # +10% if new joiner
    + (work_mode == 'On-site') * 0.05               # +5% if forced on-site
)
attrition_prob = np.clip(attrition_prob, 0, 1)      # keep probability between 0 and 1

# Generate final binary target using the above probabilities
attrition = np.random.binomial(1, attrition_prob)   # 1 = will leave, 0 = will stay


# ── Assemble Full DataFrame ───────────────

df = pd.DataFrame({
    'age'                  : age,
    'monthly_salary'       : monthly_salary,
    'years_at_company'     : years_at_company,
    'work_hours_per_week'  : work_hours_per_week,
    'performance_score'    : performance_score,
    'department'           : department,        # categorical
    'job_level'            : job_level,         # categorical
    'work_mode'            : work_mode,         # categorical
    'attrition'            : attrition          # target column
})

# Print dataset overview
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Total Records      : {df.shape[0]}")
print(f"Total Features     : {df.shape[1] - 1}")
print(f"Attrition Rate     : {df['attrition'].mean():.2%}")
print(f"Retention Rate     : {(1 - df['attrition'].mean()):.2%}")
print()
print("First 5 Rows:")
print(df.head())
print()
print("Data Types:")
print(df.dtypes)
print()



DATASET OVERVIEW
Total Records      : 1000
Total Features     : 8
Attrition Rate     : 22.50%
Retention Rate     : 77.50%

First 5 Rows:
   age  monthly_salary  years_at_company  work_hours_per_week  \
0   50           22717                18                   57   
1   36           79676                14                   54   
2   29           47952                 3                   52   
3   42          106444                12                   52   
4   40           62460                18                   36   

   performance_score   department job_level work_mode  attrition  
0                4.1  Engineering      Lead    Remote          0  
1                4.8           HR      Lead   On-site          1  
2                4.1        Sales       Mid    Remote          0  
3                3.4           HR       Mid    Remote          0  
4                4.4      Finance    Junior   On-site          0  

Data Types:
age                      int32
monthly_salary           i

In [3]:
df.shape

(1000, 9)

In [4]:
df.head()

,age,monthly_salary,years_at_company,work_hours_per_week,performance_score,department,job_level,work_mode,attrition
0,50,22717,18,57,4.1,Engineering,Lead,Remote,0
1,36,79676,14,54,4.8,HR,Lead,On-site,1
2,29,47952,3,52,4.1,Sales,Mid,Remote,0
3,42,106444,12,52,3.4,HR,Mid,Remote,0
4,40,62460,18,36,4.4,Finance,Junior,On-site,0


In [5]:
df.isnull().sum()

age                    0
monthly_salary         0
years_at_company       0
work_hours_per_week    0
performance_score      0
department             0
job_level              0
work_mode              0
attrition              0
dtype: int64

In [6]:
df.duplicated().sum()

0

In [7]:
feature_cols = [
    'age', 'monthly_salary', 'years_at_company',
    'work_hours_per_week', 'performance_score',
    'department', 'job_level', 'work_mode'
]                             # all 8 feature columns (5 numerical + 3 categorical)

X = df[feature_cols]          # feature matrix — all input columns
y = df['attrition']           # target vector — what we want to predict

In [8]:
X

,age,monthly_salary,years_at_company,work_hours_per_week,performance_score,department,job_level,work_mode
0,50,22717,18,57,4.1,Engineering,Lead,Remote
1,36,79676,14,54,4.8,HR,Lead,On-site
2,29,47952,3,52,4.1,Sales,Mid,Remote
3,42,106444,12,52,3.4,HR,Mid,Remote
4,40,62460,18,36,4.4,Finance,Junior,On-site
...,...,...,...,...,...,...,...,...
995,34,113300,4,47,3.0,Marketing,Mid,Remote
996,51,107163,7,39,1.4,Engineering,Mid,On-site
997,44,59310,13,60,1.5,Engineering,Manager,On-site
998,40,76088,12,37,4.8,Sales,Lead,Remote


In [9]:
y

0      0
1      1
2      0
3      0
4      0
      ..
995    0
996    1
997    0
998    0
999    0
Name: attrition, Length: 1000, dtype: int32

In [10]:
# ─────────────────────────────────────────
# Step 3: Identify Categorical Feature Indices
# ─────────────────────────────────────────

# CatBoost needs the COLUMN INDEX (position) of categorical features
# It does NOT need any encoding — handles categories internally by itself

categorical_features = ['department', 'job_level', 'work_mode']   # names of cat columns

cat_feature_indices = [
    feature_cols.index(col)          # get position number of each categorical column
    for col in categorical_features
]

In [11]:
cat_feature_indices

[5, 6, 7]

In [12]:
print("=" * 55)
print("CATEGORICAL FEATURE INFO")
print("=" * 55)
for name, idx in zip(categorical_features, cat_feature_indices):
    print(f"  Column: {name:<20} Index: {idx}")   # show name and its index position
print()
print("Unique values per categorical column:")
for col in categorical_features:
    print(f"  {col}: {df[col].unique().tolist()}")  # show all unique category values
print()

CATEGORICAL FEATURE INFO
  Column: department           Index: 5
  Column: job_level            Index: 6
  Column: work_mode            Index: 7

Unique values per categorical column:
  department: ['Engineering', 'HR', 'Sales', 'Finance', 'Marketing']
  job_level: ['Lead', 'Mid', 'Junior', 'Senior', 'Manager']
  work_mode: ['Remote', 'On-site', 'Hybrid']



In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,                    # feature matrix
    y,                    # target vector
    test_size=0.2,        # keep 20% data for testing
    random_state=42,      # fixed seed for reproducible split
    stratify=y            # maintain same attrition ratio in both train and test
)


In [14]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((800, 8), (200, 8), (800,), (200,))

In [15]:
model = CatBoostClassifier(
    iterations=300,           # build 300 decision trees sequentially
    learning_rate=0.05,       # each tree contributes 5% to the final prediction
    depth=6,                  # maximum depth of each tree — controls complexity
    l2_leaf_reg=5,            # L2 regularization to prevent overfitting
    random_seed=42,           # fixed seed for consistent results
    eval_metric='AUC',        # monitor AUC score during training
    loss_function='Logloss',  # log loss for binary classification
    early_stopping_rounds=30, # stop if no AUC improvement for 30 consecutive rounds
    verbose=50                # print training log every 50 iterations
)


In [16]:
model.fit(
    X_train,                           # training features
    y_train,                           # training labels
    cat_features=cat_feature_indices,  # tell CatBoost which columns are categorical
    eval_set=(X_test, y_test),         # validation set to monitor during training
    plot=False                         # dont open browser plot (keep output clean)
)


0:	test: 0.5355556	best: 0.5355556 (0)	total: 188ms	remaining: 56.2s
50:	test: 0.6192115	best: 0.6278136 (22)	total: 813ms	remaining: 3.97s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 0.6278136201
bestIteration = 22

Shrink model to first 23 iterations.


In [17]:
y_pred       = model.predict(X_test)             # predicted class labels (0 or 1)
y_pred_proba = model.predict_proba(X_test)[:, 1] # probability of attrition (class 1)

In [19]:
acc = accuracy_score(y_test, y_pred)             # fraction of correct predictions
auc = roc_auc_score(y_test, y_pred_proba)        # AUC — main metric for imbalanced data

print(f"Accuracy           : {acc:.4f} ({acc:.2%})")   # e.g. 0.8450 (84.50%)
print(f"ROC-AUC Score      : {auc:.4f}")               # e.g. 0.9012
print()
print("Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=['No Attrition', 'Attrition']))      # readable class names
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)                   # actual vs predicted table
print(f"  True Negative  (correct no-attrition) : {cm[0][0]}")
print(f"  False Positive (wrongly predicted leave): {cm[0][1]}")
print(f"  False Negative (missed actual leaves)  : {cm[1][0]}")
print(f"  True Positive  (correct attrition)     : {cm[1][1]}")
print()

Accuracy           : 0.7750 (77.50%)
ROC-AUC Score      : 0.6278

Classification Report:
              precision    recall  f1-score   support

No Attrition       0.78      1.00      0.87       155
   Attrition       0.00      0.00      0.00        45

    accuracy                           0.78       200
   macro avg       0.39      0.50      0.44       200
weighted avg       0.60      0.78      0.68       200

Confusion Matrix:
  True Negative  (correct no-attrition) : 155
  False Positive (wrongly predicted leave): 0
  False Negative (missed actual leaves)  : 45
  True Positive  (correct attrition)     : 0

